# Prepare a BioImage Archive submission from a local training set

Turn a training set's local folder into a **BioImage Archive (BIA)** submission bundle: MIFA
metadata (`Study`/`Annotations`/`Version` YAML), the BIA file lists (`file_list_images.tsv`,
`file_list_annotations.tsv` with the required `source_image` column), and the image/mask
files, all copied into one self-contained folder. No OMERO connection is needed.

For a set whose data is on OMERO, use `omero-table-to-bioimage-archive.ipynb` instead.

> Requires the optional `bia-mifa-models` package (in the `dev`/`mifa` pixi environments).

The cells below run a small self-contained demo. For real use, open your set instead:

```python
from omero_annotate_ai import TrainingSet
ts = TrainingSet.open(Path("~/omero_annotate_ai/<server>/<table_id>_<name>"))
config, rows, data_dir = ts.config, ts.units(), ts.bundle_dir
```

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile

from omero_annotate_ai import Row, create_default_config
from omero_annotate_ai.core.annotation_config import AuthorInfo
from omero_annotate_ai.core.mifa_export import save_bia_package

## 1. A demo training set

A config with the study metadata, three annotated units, and dummy image/mask files in the
layout every training set folder has (`annotation_input/`, `annotation_output/`).

In [ ]:
data_dir = Path(tempfile.mkdtemp(prefix="bia_demo_set_"))

config = create_default_config()
config.name = "nuclei_segmentation_demo"
config.study.title = "Nuclei segmentation in U2OS cells"
config.study.description = "Expert-curated nuclear segmentation masks."
config.study.keywords = ["nuclei", "segmentation", "fluorescence"]
config.study.organism = "Homo sapiens"
config.study.funding_statement = "Supported by the Example Funding Council (grant 12345)."
config.dataset.source_dataset_id = "S-BIAD123"
config.dataset.license = "CC-BY-4.0"
config.annotation_methodology.annotation_criteria = "Only in-focus nuclei were annotated."
config.annotation_methodology.annotation_method = "semi_automatic"
config.authors = [AuthorInfo(name="Jane Doe", affiliation="EMBL-EBI")]

rows = [
    Row(
        image_id=i,
        image_uri=f"omero:image:{i}",
        image_name=f"img{i}.tif",
        annotation_id=f"{i}_0_0",
        category="training" if i % 2 else "validation",
        timepoint=0,
        z_slice=0,
        channel=0,
        processed=True,
    )
    for i in (1, 2, 3)
]

(data_dir / "annotation_input").mkdir()
(data_dir / "annotation_output").mkdir()
for row in rows:
    tifffile.imwrite(data_dir / "annotation_input" / f"{row.annotation_id}.tif",
                     np.zeros((16, 16), dtype="uint16"))
    tifffile.imwrite(data_dir / "annotation_output" / f"{row.annotation_id}_mask.tif",
                     np.zeros((16, 16), dtype="uint16"))
print(f"{config.name}: {len(rows)} units in {data_dir}")

## 2. Build the BIA submission bundle

In [ ]:
bundle_dir = Path(tempfile.mkdtemp(prefix="bia_bundle_")) / "submission"
result = save_bia_package(config, rows, data_dir, bundle_dir, accession="S-BIAD123")

print("Bundle written to:", result["dir"])
print(
    f"  images: {result['n_images']} | annotations: {result['n_annotations']} "
    f"| files copied: {result['copied']} | missing: {result['missing']}"
)
print("\nBundle contents:")
for path in sorted(result["dir"].rglob("*")):
    if path.is_file():
        print("  ", path.relative_to(result["dir"]))

## 3. Inspect the file lists and MIFA metadata

In [ ]:
print("=== file_list_annotations.tsv ===")
print(pd.read_csv(result["file_list_annotations"], sep="\t").to_string(index=False))
print("\n=== file_list_images.tsv ===")
print(pd.read_csv(result["file_list_images"], sep="\t").to_string(index=False))
print("\n=== metadata/Annotations_S-BIAD123.yaml ===")
print(result["metadata"]["annotations"].read_text())

## 4. Upload to the BioImage Archive

Submit the `submission/` folder (file lists + data + `metadata/`) to the BioImage Archive
via their FTP/Aspera/Globus transfer, following
[the BIA submission guide](https://www.ebi.ac.uk/bioimage-archive/help-file-list/).
Once published you receive an accession (`S-BIAD###`) and a DOI — feed those back into
`config.dataset.source_dataset_id` / `source_dataset_url` so a Model Zoo export can
reference this dataset as the model's `training_data`.